# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

Cloning into 'FlyRank-ML'...
remote: Enumerating objects: 298, done.
remote: Counting objects: 100% (298/298), done.
remote: Compressing objects: 100% (245/245), done.
remote: Total 298 (delta 177), reused 104 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (298/298), 2.08 MiB | 13.45 MiB/s, done.
Resolving deltas: 100% (177/177), done.


In [2]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

df = pd.read_csv('/content/FlyRank-ML/data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(df.columns.tolist())

fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.
(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. Unit of analysis + time window

**Unit of analysis:** one row = one content page (`content_id`), belonging to one `client_id` (32 distinct clients). This is a **single static snapshot**, not a time series — there's no `report_date` column, no repeated rows per page. Every metric in the file is already aggregated over a trailing 90-day window ending at one export timestamp.

**Time window:** all "90-day" columns (`impressions_90d`, `ctr`, `engagement_rate`, etc.) cover the same trailing 90 days. Within that, two comparison sub-windows exist: `*_prev_30d` (days 31–60 back) and `*_last_30d` (most recent 30 days) — these are what `trend_pct` / `trend_direction` are computed from.

This matters for the contract because the unit of analysis here is flatter than the warehouse's content×day grain from the earlier draft: there's no per-day history to slice, so "time window" means picking which pre-aggregated column families are safe to use, not picking a calendar range.

## 2. Fields: feature / label / context / excluded

*Sort every field into these four buckets. Excluded needs a why.*

**Label**
| Field | Why |
|---|---|
| `is_declining_label` (derived: `trend_direction == 'down'`) | The target for ML-03's binary classification task. Built from `trend_direction`, which is itself computed from `impressions_last_30d` vs `impressions_prev_30d`. |

**Features — safe (available before/independent of the label's outcome window)**

Numeric:
| Field | Note |
|---|---|
| `content_age_days` | static content property |
| `days_since_last_update` | static content property |
| `word_count`, `char_count` | static; ~7,699 rows blank (see missingness check below) |
| `search_volume`, `competition`, `cpc` | keyword-context; blank when no keyword data (~2,468 rows) |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | days 31–60 back — precedes the label's outcome window (last 30 days), so safe |

Categorical:
| Field | Note |
|---|---|
| `content_type` | keyword/feedly/comparison article |
| `main_intent` | blank when unknown |
| `competition_level` | LOW/MEDIUM/HIGH |
| `age_tier`, `freshness_tier` | transparent buckets of the two age/update fields above |
| `word_count_tier`, `char_count_tier` | transparent buckets of word/char count |

**Excluded — window overlap with the label (leakage, gate 2)**

| Field | Why excluded |
|---|---|
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | This *is* the outcome window the label is computed from — using it as a feature would mean the model is fed the answer. |
| `impressions_90d`, `clicks_90d`, `sessions_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions` | Every `_90d` total is `prev_30d + last_30d + remainder` (confirmed in ML-02) — it necessarily includes the same last-30-day window the label is built from. Dropped on principle (gate 2: not cleanly available before the outcome window), not only where correlation happens to be high. |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | All derived from the `_90d` totals above, so they inherit the same window-overlap problem one level removed. |
| `impression_tier`, `position_tier` | Transparent buckets, but built from the excluded `_90d`/`avg_position` columns — excluded for the same reason as their source columns. |

**Excluded — label source (leakage, gate 1)**

| Field | Why excluded |
|---|---|
| `trend_direction`, `trend_pct` | These define the label directly. Using them as features would be circular by construction. |

**Context — not features, not the label, but used elsewhere**

| Field | Role |
|---|---|
| `content_id`, `client_id` | Join keys and grouped-split keys only — never model inputs |
| `provider_used`, `model_used` | Explicitly flagged "not a model feature" in the data dictionary |
| `impressions_90d`, `sessions_90d` | Excluded as *features*, but reused as the **eligibility filter** (volume floor from ML-02: `impressions_90d >= 100` and `sessions_90d > 0`). This is a population-scoping decision made before modeling, not a feature fed into the model — worth stating explicitly so the two roles don't get confused later.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 - Grain Check**

In [3]:
print("Total rows:", len(df))
print("Unique content_id:", df['content_id'].nunique())
print("Unique client_id:", df['client_id'].nunique())
assert len(df) == df['content_id'].nunique(), "content_id is not a unique row key!"

Total rows: 30000
Unique content_id: 30000
Unique client_id: 32


Confirms one row really is one content×day — same check as before, rerun on the new slice.

**Query 2 — Label distribution and the `_90d`/`_prev_30d`/`_last_30d` relationship**

In [4]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df['is_declining_label'].value_counts(normalize=True))
print()
# Confirm the window-overlap claim used to justify excluding _90d columns
check = df[['impressions_90d', 'impressions_prev_30d', 'impressions_last_30d']].copy()
check['sum_30d'] = check['impressions_prev_30d'] + check['impressions_last_30d']
check['diff'] = check['impressions_90d'] - check['sum_30d']
print(check['diff'].describe())

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64

count     30000.000000
mean       1988.229067
std        6099.411747
min           0.000000
25%          28.000000
50%         308.000000
75%        1512.250000
max      258505.000000
Name: diff, dtype: float64


If `diff` is mostly small/positive (a remainder from the days 61–90 window), it confirms `impressions_90d` is a rollup that includes `impressions_last_30d` — supporting the leakage exclusion above rather than just asserting it.

**Query 3 — Volume floor / eligibility filter**

In [5]:
eligible = df[(df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)]
print(f"Eligible: {len(eligible)} / {len(df)} ({len(eligible)/len(df):.1%})")
print()
print("Label balance, full dataset:")
print(df['is_declining_label'].value_counts(normalize=True))
print()
print("Label balance, eligible-only:")
print(eligible['is_declining_label'].value_counts(normalize=True))

Eligible: 22006 / 30000 (73.4%)

Label balance, full dataset:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64

Label balance, eligible-only:
is_declining_label
1    0.597655
0    0.402345
Name: proportion, dtype: float64


Same check introduced in ML-02, rerun here as part of the formal contract rather than as exploratory code — confirms whether the eligibility filter meaningfully shifts the label balance before the model ever sees the data.

**Query 4 — Client concentration**

In [6]:
client_counts = df['client_id'].value_counts()
print(client_counts.describe())
print()
print("Top 5 clients by row count:")
print(client_counts.head(5))
print()
top5_share = client_counts.head(5).sum() / len(df)
print(f"Top 5 clients: {top5_share:.1%} of all rows")

count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64

Top 5 clients by row count:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
Name: count, dtype: int64

Top 5 clients: 56.8% of all rows


Same concentration risk flagged in ML-02: any train/test split must group by `client_id`, not split rows randomly.

Per the data dictionary, keyword-context missingness (`search_volume`, `competition`, `cpc`) tracks `content_type` — `feedly article` rows carry no keyword data at all. That's a structural pattern, not random dropout, so a blind `fillna(0)` would silently encode content type into the features. Confirm this holds before choosing an imputation strategy.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Single snapshot, not a time series.** There's no way to observe a genuine future outcome from this file alone — `trend_direction` is itself a proxy computed within one 90-day window, not a decline confirmed by watching what actually happened afterward. Any claim from this lane stays in "current-window pattern," not "forecast."
- **Small client base, unevenly concentrated.** 32 clients total, with a handful contributing a disproportionate share of rows (confirmed in Query 4). Findings may reflect a few clients' content strategies more than a general pattern — this is why grouped, not random, validation matters, and why any strong claim should be checked for stability with the dominant clients held out.
- **Structural missingness tied to content type.** Keyword-context and word/char-count fields go blank in patterns that follow `content_type`, not at random (Query 5). Treated wrong, this becomes silent feature leakage of content type through the back door of "0 vs. present."
- **No article text.** Only metadata and tiers are available — nothing here supports claims about *why* a page is declining at the content level, only *that* it correlates with certain measurable signals.
- **No causal claim possible.** This data shows association between current signals and the current-window decline label. It cannot show that refreshing a flagged page would cause recovery — that would require an actual experiment.

## 5. Output

*What does this analysis hand to the human, in one sentence?*

A ranked review queue of eligible content pages (those passing the volume floor), ordered by predicted probability of `is_declining_label`, each with reason codes explaining why it was flagged — for a content team to review and decide on, not an automated refresh trigger.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.